In [ ]:
%reset -f
from sqlite3 import connect
from glob import glob
from ipynbname import name
from collections import Counter
from os import path
import numpy as np
import torch
from torch.nn import *
class Res(Sequential):
    def forward(s, x):
        return super().forward(x)+x
class At(MultiheadAttention):
    def forward(s, x):
        return super().forward(x, x, x, attn_mask=Transformer.generate_square_subsequent_mask(x.size(-2),dtype=x.dtype,device=x.device), is_causal=True)[0]
class Swiglu(Linear):
    def __init__(s, i, o=0):
        super().__init__(i, 2*(o or i//6*16))
        s._ = Linear(o or i//6*16, i)
    def forward(s, x):
        x,_ = super().forward(x).chunk(2,-1)
        return s._(x*functional.silu(_))
class REnCoder(Sequential):
    def __init__(s, o, i, bin, *emb, drop=.1):
        super().__init__(Linear(i,o),EmbeddingBag(bin,o,scale_grad_by_freq=True,mode='sum'),*(Embedding(i,o,0,scale_grad_by_freq=True)for i in emb))
        s.drop = drop
    def forward(s, x):
        x = x.split((s[0].in_features,s[1].num_embeddings//64,*[1]*(len(s)-2)),-1)
        return sum(
            (s[i](drop(x[...,0],s.drop,s.training))for i,x in enumerate(x[2:],2)),
            s[0](functional.dropout(x[0].to(s[0].weight.dtype),s.drop,s.training,True))+
            s[1](
                (_:=drop((x[1][...,None]&1<<torch.arange(64,device=x[0].device)).flatten(0,-3).flatten(1),s.drop,s.training).nonzero())[:,1],
                torch.zeros(1+x[0].shape[:-1].numel(),dtype=torch.int,device=x[0].device)[1:].scatter_add_(
                    0,
                    _[:,0],
                    torch.ones(1,dtype=torch.int,device=x[0].device).expand(len(_))
                )._base[:-1].cumsum(0)
            ).unflatten(0,x[0].shape[:-1])
        )
def drop(a, p, train):
    return a*(torch.randint(256,a.shape,dtype=torch.uint8,device=a.device)>int(256*p))if train else a
def pack(*a):
    x = 1
    for a in a:
        x <<= a[-1]
        for a in a[:-1]:
            x |= a if isinstance(a,int)else 1<<int(a)-1 if a else 0
    while x>1:
        yield x&0xffffffffffffffff
        x >>= 64
def randsplit(x):
    x,a = x.T
    a = a+1
    b = torch.randint_like(a,256,dtype=torch.uint8).fmod_(a)
    a -= b
    c = torch.randint_like(a,256,dtype=torch.uint8).fmod_(a)
    a -= c
    a -= 1
    return torch.cat([torch.stack([x,a],1)[a.bool()],torch.stack([x,b],1)[b.bool()],torch.stack([x,c],1)[c.bool()]])
CARD = {id:(
    max(atk,0),
    0 if type&0x4000000 else max(Def,0),
    level&15,
    level>>24,
    len(name),
    len(desc),
    len(_:=path.exists(_:=f'script/c{alias or id}.lua')and open(_).read()or''),
    *pack(
        (ot-(ot>0x4)*0x4,3),
        (*map(float,(setcode&0xfff,setcode>>16&0xfff,setcode>>32&0xfff,setcode>>48&0xfff)),475),
        ((setcode>>12|setcode>>28|setcode>>44|setcode>>60)&0xf,4),
        (type,27),
        (category%(1<<32),32),
        (Def-(Def>0x10)*0x10 if type&0x4000000 else 0,8),
        (sum((j in name)<<i for i,j in enumerate('- 星者魔生之战王光兽火灵剑人神骑女大天机士物子龙黑的·')),28),
        (sum((j in desc)<<i for i,j in enumerate('手把·攻「表方对时有不存个那果外为次墓在地坏殊从名特组用击破')),30),
        (sum((j in _)<<i for i,j in enumerate(
            'not GetLocationCount IsType IsLocation IsExistingMatchingCard IsFaceup tc target EFFECT_FLAG_CARD_TARGET false filter true POS_FACEUP SetValue LOCATION_GRAVE LOCATION_HAND REASON_EFFECT e3 EVENT_FREE_CHAIN '
            'SetReset EFFECT_TYPE_SINGLE EFFECT_TYPE_TRIGGER_O Card EFFECT_TYPE_ACTIVATE CATEGORY_SPECIAL_SUMMON SetCondition SetCost SetCountLimit SelectMatchingCard EFFECT_TYPE_FIELD GetCount IsRelateToEffect IsSetCard'.split()
        )),33),
    ),
    race.bit_length(),
    attribute.bit_length(),
    alias or id,
)for id,ot,alias,setcode,type,atk,Def,level,race,attribute,category,name,desc in connect('cards.cdb').execute('select datas.*,name,desc from datas join texts on datas.id=texts.id')}
CARD[0] = (0,)*20

In [ ]:
RETRAIN = False
try:
    MAP = dict(map(reversed,enumerate(MAP_:=np.fromfile(name()+'.i','i').tolist())))
except FileNotFoundError as _:
    RETRAIN = True
    MAP = {0:0}
DECK = [
    torch.tensor(list(Counter(i).items()),dtype=torch.short,device='cuda')
    for i in glob('deck/**/*.ydk',recursive=True,include_hidden=True)
    for i in[[
        MAP.setdefault(CARD[int(i)][-1],len(MAP))
        for i in open(i).read().split('!')[0].split()
        if i.isdigit()and int(i)and int(i)in(RETRAIN and CARD or MAP)
    ]]
    if len(i)>=40
]
CARD = torch.tensor([CARD[i][:-1]for i in MAP],dtype=torch.uint64,device=DECK[0].device).view(torch.int64)
np.array(list(MAP),'i').tofile(name()+'.i')
try:
    if RETRAIN:
        raise FileNotFoundError
    m,o,s = torch.load(name()+'.pk',CARD.device,weights_only=False)
except FileNotFoundError as _:
    m = 256
    m = torch.compile(Sequential(
        REnCoder(m, 8, 640, 27,8,len(MAP)),
        Dropout(.1),
        *(Sequential(
            Res(
                LayerNorm(m),
                At(m, 32, batch_first=True),
                Dropout(.1),
            ),
            Res(
                LayerNorm(m),
                Swiglu(m),
                Dropout(.1),
            ),
        )for _ in range(5)),
        Linear(m, len(MAP))
    ).to(CARD.device))
    o = torch.optim.AdamW((
        {'params':[i for i in m.parameters()if i.squeeze().ndim>1],'weight_decay':1e-4},
        {'params':[i for i in m.parameters()if i.squeeze().ndim<2],'weight_decay':0}
    ),.0007)
    s = torch.amp.GradScaler()

In [ ]:
while True:
    for _ in range(20):
        for _ in range(20):
            L = 0
            for _ in range(20000):
                o.zero_grad()
                a = [randsplit(DECK[i])for i in torch.randint(len(DECK),[8])]
                x = torch.stack([torch.constant_pad_nd(i,(0,0,0,max(map(len,a))-len(i)))for i in a])
                x = x.gather(1,torch.randint(256,x.shape[:2],dtype=torch.uint8,device=x.device).argsort()[:,:,None].expand(x.shape))
                x = torch.cat([x[:,:,1:],CARD[x[:,:,0].int()],x[:,:,:1]],2)
                with torch.amp.autocast(CARD.device.type, torch.float16):
                    y = m._orig_mod(x).log_softmax(2)
                    p = torch.zeros_like(y).scatter_add_(2,x[:,:,-1:],x[:,:,:1].to(y.dtype))
                    cump = p[:,1:].flip(1).cumsum_(1).flip(1)
                    cumask = cump.sign().log_()
                    cumask[cumask.all(2),0] = 0
                    cump[~cump.any(2),0] = 1
                    cump /= -cump.numel()*cump.sum(2,True)
                    p = p.sum(1)
                    mask = p.sign().log_()
                    p /= -p.numel()*p.sum(1,True)
                    l = y.mean(1).ravel()@p.ravel(), y[:,:-1].ravel()@cump.ravel(), -(y[:,:-1]+cumask).logsumexp(2).mean(), -(y[:,:-1]+mask[:,None]).logsumexp(2).mean()
                s.scale(l[1]).backward()
                s.step(o)
                s.update()
                L += torch.stack([i.data for i in l]).float()
            print(L)
            torch.save((m,o,s),name()+'.pk')
        torch.save((m,o,s),name()+'.0pk')
    torch.save((m,o,s),name()+'.1pk')